In [0]:
!pip install kagglehub[pandas-datasets]

In [0]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "customer_support_tickets_200k.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mirzayasirabdullah07/customer-support-tickets-dataset-200k-records",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

In [0]:
df.info()

### Check for Null Values

In [0]:
# check for na values
df[df.isna().any(axis=1)]

In [0]:
# fill na values
df['browser'] = df['browser'].fillna("No browser")
df

## Preparing Data for Pipeline

#### Remove Noise from Text and Normalize

In [0]:
import re

# text normalization 
def normalize_text(text):
    """
    Standardizes text for the agent to read better
    """
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text
df['issue_description'] = df['issue_description'].apply(normalize_text)
df['resolution_notes'] = df['resolution_notes'].apply(normalize_text)
df.head()

#### Map Categorical Columns to Integers

In [0]:
import pandas as pd

# columns that should be categorized
cat_cols = ['category', 'priority', 'status', 'escalated']

# create a new column with int values for each category 
# for each column listed above
for col in cat_cols:
    numerical_values, _ = pd.factorize(df[col])
    current_index = df.columns.get_loc(col)
    new_col_name = f'{col}_id'
    df.insert(current_index + 1, column = new_col_name, value = numerical_values)

# convert date columns to datetime
df['ticket_created_date'] = pd.to_datetime(df['ticket_created_date'])
df['ticket_resolved_date'] = pd.to_datetime(df['ticket_resolved_date'])

In [0]:
# check data types for each column
df.info()

#### Create Embeddings 

In [0]:
pip install sentence-transformers

In [0]:
# create embeddings with batch processing
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['issue_description'].tolist()

# Process in batches with progress bar
batch_size = 1000
embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)

df['embeddings'] = list(embeddings)
print(f"Created embeddings with shape: {embeddings.shape}")


In [0]:
df.head()